In [1]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """

    # Get default branch. it might not be the default main.
    api_url = f"https://api.github.com/repos/{repo_owner}/{repo_name}"
    repo_info = requests.get(api_url).json()
    default_branch = repo_info["default_branch"]
    
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/{default_branch}'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    target_keywords = (
        'model',
        'incremental',
        'ref',
        'source',
        'seed',
        'snapshot',
        'macro',
        'materialization'
    )

    exclude_keywords = (
        'reference',
        'command',
        'api',
        'changelog',
        'release',
        'faq'
    )    

    exclude_paths = (
        '/blog/',
        '/contributing/',
        '/release-notes/',
        '/cloud/',
        '/snippets/',
        '/community/'        
    )    
    
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()
       
        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue

        #
        # Filter to target beginner-friendly folders
        #

        # keyword filter
        if not any(keyword in filename_lower for keyword in target_keywords):
            continue       

        if any(k in filename_lower for k in exclude_keywords):
            continue       

        if any(ex in filename_lower for ex in exclude_paths):
            continue            

        # skip deep reference docs
        if len(filename_lower.split('/')) > 6:
            continue            
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data


In [2]:
dbt_docs = read_repo_data('dbt-labs', 'docs.getdbt.com')

print(f"FAQ documents: {len(dbt_docs)}")

FAQ documents: 34


In [31]:
for item in dbt_docs:
    print(item.get('filename'))

docs.getdbt.com-current/website/docs/best-practices/clone-incremental-models.md
docs.getdbt.com-current/website/docs/best-practices/how-we-build-our-metrics/semantic-layer-3-build-semantic-models.md
docs.getdbt.com-current/website/docs/best-practices/how-we-build-our-metrics/semantic-layer-8-refactor-a-rollup.md
docs.getdbt.com-current/website/docs/best-practices/how-we-handle-real-time-data/2-incremental-patterns.md
docs.getdbt.com-current/website/docs/best-practices/how-we-style/1-how-we-style-our-dbt-models.md
docs.getdbt.com-current/website/docs/best-practices/materializations/materializations-guide-1-guide-overview.md
docs.getdbt.com-current/website/docs/best-practices/materializations/materializations-guide-2-available-materializations.md
docs.getdbt.com-current/website/docs/best-practices/materializations/materializations-guide-3-configuring-materializations.md
docs.getdbt.com-current/website/docs/best-practices/materializations/materializations-guide-4-incremental-models.md
doc

In [32]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result


In [4]:
dbt_chunks = []

for doc in dbt_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    dbt_chunks.extend(chunks)

print(f"dbt chunks: {len(dbt_chunks)}")

dbt chunks: 308


In [5]:
import re
text = dbt_docs[9]['content']
paragraphs = re.split(r"\n\s*\n", text.strip())

print(paragraphs)

['First, let’s consider some properties of various levels of our dbt project and materializations.', '- 🔍\xa0**Views** return the freshest, real-time state of their input data when they’re queried, this makes them ideal as **building blocks** for larger models.\n  - 🧶\xa0 When we’re building a model that stitches lots of other models together, we don’t want to worry about all those models having different states of freshness because they were built into tables at different times. We want all those inputs to give us all the underlying source data available.\n- 🤏\xa0**Views** are also great for **small datasets** with minimally intensive logic that we want **near realtime** access to.\n- 🛠️\xa0**Tables** are the **most performant** materialization, as they just return the transformed data when they’re queried, with no need to reprocess it.\n  - 📊\xa0 This makes tables great for **things end users touch**, like a mart that services a popular dashboard.\n  - 💪\xa0Tables are also ideal for 

In [6]:
import re

def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.
    
    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)
    
    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)
    
    return sections



In [7]:
dbt_chunks = []

for doc in dbt_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        dbt_chunks.append(section_doc)

print(f"dbt chunks: {len(dbt_chunks)}")        


dbt chunks: 100


In [8]:
from openai import OpenAI

openai_client = OpenAI()


def llm(prompt, model='gpt-4o-mini'):
    messages = [
        {"role": "user", "content": prompt}
    ]

    response = openai_client.responses.create(
        model='gpt-4o-mini',
        input=messages
    )

    return response.output_text


In [9]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [10]:
def intelligent_chunking(text):
    prompt = prompt_template.format(document=text)
    response = llm(prompt)
    sections = response.split('---')
    sections = [s.strip() for s in sections if s.strip()]
    return sections

In [11]:
'''
from tqdm.auto import tqdm

dbt_docs_chunks = []

for doc in tqdm(dbt_docs):
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')

    sections = intelligent_chunking(doc_content)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        dbt_docs_chunks.append(section_doc)
'''

"\nfrom tqdm.auto import tqdm\n\ndbt_docs_chunks = []\n\nfor doc in tqdm(dbt_docs):\n    doc_copy = doc.copy()\n    doc_content = doc_copy.pop('content')\n\n    sections = intelligent_chunking(doc_content)\n    for section in sections:\n        section_doc = doc_copy.copy()\n        section_doc['section'] = section\n        dbt_docs_chunks.append(section_doc)\n"

In [12]:
dbt_chunks = []

for doc in dbt_docs:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    dbt_chunks.extend(chunks)

print(f"dbt chunks: {len(dbt_chunks)}")

dbt chunks: 308


In [15]:
from minsearch import Index

dbt_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

dbt_index.fit(dbt_chunks)


In [16]:
query = 'What is a dbt model and how does it work?'
results = dbt_index.search(query)

print(results)


[{'start': 0, 'chunk': '## Fields and model names\n\n- 👥 Models should be pluralized, for example, `customers`, `orders`, `products`.\n- 🔑 Each model should have a primary key.\n- 🔑 The primary key of a model should be named `<object>_id`, for example, `account_id`. This makes it easier to know what `id` is being referenced in downstream joined models.\n- Use underscores for naming dbt models; avoid dots.\n  - ✅  `models_without_dots`\n  - ❌ `models.with.dots`\n  - Most data platforms use dots to separate `database.schema.object`, so using underscores instead of dots reduces your need for [quoting](/reference/resource-properties/quoting) as well as the risk of issues in certain parts of <Constant name="dbt" />. For more background, refer to [this GitHub issue](https://github.com/dbt-labs/dbt-core/issues/3246).\n- 🔑 Keys should be string data types.\n- 🔑 Consistency is key! Use the same field names across models where possible. For example, a key to the `customers` table should be named

In [17]:
from tqdm.auto import tqdm
import numpy as np
from minsearch import VectorSearch

from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

dbt_embeddings = []

for d in tqdm(dbt_chunks):
    v = embedding_model.encode(d['chunk'])
    dbt_embeddings.append(v)

dbt_embeddings = np.array(dbt_embeddings)

dbt_vindex = VectorSearch()
dbt_vindex.fit(dbt_embeddings, dbt_chunks)


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

  0%|          | 0/308 [00:00<?, ?it/s]

In [18]:
query = 'What is a dbt model and how does it work?'
q = embedding_model.encode(query)
results = dbt_vindex.search(q)
print(results)

[{'start': 27000, 'chunk': "s://github.com/dbt-labs/dbt-core/discussions/5738)\n:::\n\n## Limitations\n\nPython models have capabilities that SQL models do not. They also have some drawbacks compared to SQL models:\n\n- **Time and cost.** Python models are slower to run than SQL models, and the cloud resources that run them can be more expensive. Running Python requires more general-purpose compute. That compute might sometimes live on a separate service or architecture from your SQL models. **However:** We believe that deploying Python models via dbt—with unified lineage, testing, and documentation—is, from a human standpoint, **dramatically** faster and cheaper. By comparison, spinning up separate infrastructure to orchestrate Python transformations in production and different tooling to integrate with dbt is much more time-consuming and expensive.\n- **Syntax differences** are even more pronounced. Over the years, dbt has done a lot, via dispatch patterns and packages such as `dbt_u

In [19]:
query = 'What is a dbt model and how does it work?'

text_results = dbt_index.search(query, num_results=5)

q = embedding_model.encode(query)
vector_results = dbt_vindex.search(q, num_results=5)

final_results = text_results + vector_results

print(final_results)


[{'start': 0, 'chunk': '## Fields and model names\n\n- 👥 Models should be pluralized, for example, `customers`, `orders`, `products`.\n- 🔑 Each model should have a primary key.\n- 🔑 The primary key of a model should be named `<object>_id`, for example, `account_id`. This makes it easier to know what `id` is being referenced in downstream joined models.\n- Use underscores for naming dbt models; avoid dots.\n  - ✅  `models_without_dots`\n  - ❌ `models.with.dots`\n  - Most data platforms use dots to separate `database.schema.object`, so using underscores instead of dots reduces your need for [quoting](/reference/resource-properties/quoting) as well as the risk of issues in certain parts of <Constant name="dbt" />. For more background, refer to [this GitHub issue](https://github.com/dbt-labs/dbt-core/issues/3246).\n- 🔑 Keys should be string data types.\n- 🔑 Consistency is key! Use the same field names across models where possible. For example, a key to the `customers` table should be named

In [20]:
def text_search(query):
    return dbt_index.search(query, num_results=5)

def vector_search(query):
    q = embedding_model.encode(query)
    return dbt_vindex.search(q, num_results=5)

def hybrid_search(query):
    text_results = text_search(query)
    vector_results = vector_search(query)
    
    # Combine and deduplicate results
    seen_ids = set()
    combined_results = []

    for result in text_results + vector_results:
        if result['filename'] not in seen_ids:
            seen_ids.add(result['filename'])
            combined_results.append(result)
    
    return combined_results


In [21]:
import openai

openai_client = openai.OpenAI()

# user_prompt = "I just discovered (DBT) by chance, can I know more about it?"
#user_prompt = "How do I know if DBT is right for me?"
user_prompt = "What is this knowledgebase?"

chat_messages = [
    {"role": "user", "content": user_prompt}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
)

print(response.output_text)


This knowledge base is a collection of information and resources designed to assist users with a wide range of topics. It encompasses various subjects, including science, technology, history, culture, and more. The goal is to provide accurate, informative, and helpful responses to inquiries. If you have a specific question or topic in mind, feel free to ask!


In [22]:
text_search_tool = {
    "type": "function",
    "name": "text_search",
    "description": "Search the DBT database",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the DBT."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


In [23]:
#system_prompt = """
#You are a helpful assistant. 
#"""

system_prompt = """
You are a helpful assistant for a course.
You MUST use the text_search_tool to answer questions.
Do not answer from your own knowledge.
If the question is about DBT, search the documents first.
"""

question = "What is a DBT model and how does it work?"

chat_messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": question}
]

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)

print(response)

Response(id='resp_013b3bb0e6513deb0069d0a5f8dc988194a621ae87d7dcd0c7', created_at=1775281656.0, error=None, incomplete_details=None, instructions=None, metadata={}, model='gpt-4o-mini-2024-07-18', object='response', output=[ResponseFunctionToolCall(arguments='{"query":"DBT model definition and functionality"}', call_id='call_74MKN0dcEprAhnScnq24gJYz', name='text_search', type='function_call', id='fc_013b3bb0e6513deb0069d0a5f97d708194aa137d4534508d5a', namespace=None, status='completed')], parallel_tool_calls=True, temperature=1.0, tool_choice='auto', tools=[FunctionTool(name='text_search', parameters={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query text to look up in the DBT.'}}, 'required': ['query'], 'additionalProperties': False}, strict=True, type='function', defer_loading=None, description='Search the DBT database')], top_p=1.0, background=False, completed_at=1775281657.0, conversation=None, max_output_tokens=None, max_tool_calls=None, pre

In [24]:
print(response.output[0])

ResponseFunctionToolCall(arguments='{"query":"DBT model definition and functionality"}', call_id='call_74MKN0dcEprAhnScnq24gJYz', name='text_search', type='function_call', id='fc_013b3bb0e6513deb0069d0a5f97d708194aa137d4534508d5a', namespace=None, status='completed')


In [25]:
import json

call = response.output[0]

arguments = json.loads(call.arguments)
result = text_search(**arguments)

call_output = {
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": json.dumps(result),
}



In [26]:
chat_messages.append(call)
chat_messages.append(call_output)

response = openai_client.responses.create(
    model='gpt-4o-mini',
    input=chat_messages,
    tools=[text_search_tool]
)

print(response.output_text)

A **DBT model** is a SQL file that defines how to transform raw data into a structured format that can be consumed by BI tools and analysts. DBT operates on the principle of data transformation within your data warehouse, allowing you to create cleaner and more organized datasets.

Here's how it works:

1. **Model Creation**: You define a model using SQL queries. These queries can either create tables or views within the data warehouse.
  
2. **Materialization**: DBT supports different types of materialization strategies:
   - **Table**: Creates a new table.
   - **View**: Creates a view over existing tables without storing data.
   - **Incremental**: Updates an existing table with only new data, which can significantly improve performance.

3. **Dependency Management**: DBT automatically manages dependencies between models, ensuring that they are built in the correct order.

4. **Execution**: You can run DBT commands to build models. DBT compiles your SQL files and executes them again

In [27]:
from typing import List, Any

def text_search(query: str) -> List[Any]:
    """
    Perform a text-based search on the DBT index.

    Args:
        query (str): The search query string.

    Returns:
        List[Any]: A list of up to 5 search results returned by the DBT index.
    """
    return dbt_index.search(query, num_results=5)

In [28]:
from pydantic_ai import Agent

agent = Agent(
    name="dbt_agent",
    instructions=system_prompt,
    tools=[text_search],
    model='gpt-4o-mini'
)


In [29]:
question = "What is a DBT model and how does it work?"
#question = "What is the difference between a view and a table in dbt?"

result = await agent.run(user_prompt=question)

In [30]:
print(result.output)

A **DBT model** refers to a SQL file that encapsulates a transformation logic within the dbt (data build tool) framework. Here’s how it works and its essential features:

1. **Structure**: 
    - DBT models should be named using pluralization (e.g., `customers`, `orders`).
    - Each model typically contains a primary key, helpful for identifying records (e.g., `customer_id`).
    - Column names should be descriptive and use `snake_case` format (e.g., `order_total`, `created_at`).

2. **Transformation Logic**: 
    - A common structure for a DBT model involves using Common Table Expressions (CTEs) to define the transformation logic, starting from source data. For example:
      ```sql
      with source as (
          select * from {{ source('ecom', 'raw_orders') }}
      ),
      renamed as (
          select
              id as order_id,
              store_id as location_id,
              (order_total / 100.0)::float as order_total,
              ...
          from source
      )
   

In [33]:
from pydantic_ai.messages import ModelMessagesTypeAdapter


def log_entry(agent, messages, source="user"):
    tools = []

    for ts in agent.toolsets:
        tools.extend(ts.tools.keys())

    dict_messages = ModelMessagesTypeAdapter.dump_python(messages)

    return {
        "agent_name": agent.name,
        "system_prompt": agent._instructions,
        "provider": agent.model.system,
        "model": agent.model.model_name,
        "tools": tools,
        "messages": dict_messages,
        "source": source
    }


In [34]:
import json
import secrets
from pathlib import Path
from datetime import datetime


LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)


def serializer(obj):
    if isinstance(obj, datetime):
        return obj.isoformat()
    raise TypeError(f"Type {type(obj)} not serializable")


def log_interaction_to_file(agent, messages, source='user'):
    entry = log_entry(agent, messages, source)

    ts = entry['messages'][-1]['timestamp']
    # ts_obj = datetime.fromisoformat(ts.replace("Z", "+00:00"))
    ts_obj = ts
    ts_str = ts_obj.strftime("%Y%m%d_%H%M%S")
    rand_hex = secrets.token_hex(3)

    filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
    filepath = LOG_DIR / filename

    with filepath.open("w", encoding="utf-8") as f_out:
        json.dump(entry, f_out, indent=2, default=serializer)

    return filepath


In [35]:
system_prompt = """
You are a helpful assistant for a course.  

Use the search tool to find relevant information from the course materials before answering questions.  

If you can find specific information through search, use it to provide accurate answers.

Always include references by citing the filename of the source material you used.  
When citing the reference, replace "docs.getdbt.com-current" by the full path to the GitHub repository: "https://github.com/dbt-labs/docs.getdbt.com/blob/current/"
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.  
""".strip()

# Create another version of agent, let's call it dbt_agent_v2
agent = Agent(
    name="dbt_agent_v2",
    instructions=system_prompt,
    tools=[text_search],
    model='gpt-4o-mini'
)


In [36]:
question = input()
result = await agent.run(user_prompt=question)
print(result.output)
log_interaction_to_file(agent, result.new_messages())


 What is a dbt model and how does it work?


A **dbt model** is a fundamental concept in dbt (data build tool) used for transforming data in your data warehouse. It represents a single SQL file that contains a query, allowing you to define a transformation from raw data into a more refined structure that can be easily utilized for analysis and reporting. 

### How dbt Models Work

1. **Materialization**: dbt models can be materialized in several ways:
   - **Table**: The result of the model is stored as a table in the data warehouse.
   - **View**: The model is stored as a view that dynamically queries the underlying data.
   - **Incremental**: Only new or updated data is processed, improving efficiency for large datasets.

2. **SQL Files**: Each model is defined in a `.sql` file within a `models` directory of a dbt project. The SQL file includes the transformation logic; when dbt runs, it executes the SQL query contained in the model file.

3. **Dependencies**: Models can have dependencies on one another. For example, a model ca

PosixPath('logs/dbt_agent_v2_20260404_064329_261321.json')

In [37]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).
We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met. 

Checklist:

- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do  
- answer_relevant: The response directly addresses the user's question  
- answer_clear: The answer is clear and correct  
- answer_citations: The response includes proper citations or sources when required  
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked? 

Output true/false for each check and provide a short explanation for your judgment.
""".strip()


In [38]:
from pydantic import BaseModel

class EvaluationCheck(BaseModel):
    check_name: str
    justification: str
    check_pass: bool

class EvaluationChecklist(BaseModel):
    checklist: list[EvaluationCheck]
    summary: str


In [39]:
eval_agent = Agent(
    name='eval_agent',
    model='gpt-5-nano',
    instructions=evaluation_prompt,
    output_type=EvaluationChecklist
)


In [40]:
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [41]:
def load_log_file(log_file):
    with open(log_file, 'r') as f_in:
        log_data = json.load(f_in)
        log_data['log_file'] = log_file
        return log_data


In [42]:
log_record = load_log_file('./logs/dbt_agent_v2_20260404_064329_261321.json')

instructions = log_record['system_prompt']
question = log_record['messages'][0]['parts'][0]['content']
answer = log_record['messages'][-1]['parts'][0]['content']
log = json.dumps(log_record['messages'])

user_prompt = user_prompt_format.format(
    instructions=instructions,
    question=question,
    answer=answer,
    log=log
)


In [43]:
result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)

checklist = result.output
print(checklist.summary)

for check in checklist.checklist:
    print(check)

Tool call initiated to fetch course-material definitions of dbt model.
check_name='instructions_follow' justification="The assistant is about to perform a tool search as instructed; will follow the user's instruction to search before answering." check_pass=True
check_name='instructions_avoid' justification='No disallowed content; following allowed actions.' check_pass=True
check_name='tool_call_search' justification='Tool call will be performed in this turn.' check_pass=True
check_name='answer_relevant' justification="Pending tool results; final answer will address the user's question." check_pass=False
check_name='answer_clear' justification='Pending tool results; final answer not yet provided.' check_pass=False
check_name='answer_citations' justification='Will include citations in final answer.' check_pass=False
check_name='completeness' justification='Will ensure coverage of concept; pending results.' check_pass=False
check_name='instruments_follow' justification='Already invoked to

In [44]:
def simplify_log_messages(messages):
    log_simplified = []

    for m in messages:
        parts = []
    
        for original_part in m['parts']:
            part = original_part.copy()
            kind = part['part_kind']
    
            if kind == 'user-prompt':
                del part['timestamp']
            if kind == 'tool-call':
                del part['tool_call_id']
            if kind == 'tool-return':
                del part['tool_call_id']
                del part['metadata']
                del part['timestamp']
                # Replace actual search results with placeholder to save tokens
                part['content'] = 'RETURN_RESULTS_REDACTED'
            if kind == 'text':
                del part['id']
    
            parts.append(part)
    
        message = {
            'kind': m['kind'],
            'parts': parts
        }
    
        log_simplified.append(message)
    return log_simplified


In [45]:
async def evaluate_log_record(eval_agent, log_record):
    messages = log_record['messages']

    instructions = log_record['system_prompt']
    question = messages[0]['parts'][0]['content']
    answer = messages[-1]['parts'][0]['content']

    log_simplified = simplify_log_messages(messages)
    log = json.dumps(log_simplified)

    user_prompt = user_prompt_format.format(
        instructions=instructions,
        question=question,
        answer=answer,
        log=log
    )

    result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)
    return result.output 


log_record = load_log_file('./logs/dbt_agent_v2_20260404_064329_261321.json')
eval1 = await evaluate_log_record(eval_agent, log_record)


In [51]:
print(eval1)

checklist = eval1
print(checklist.summary)

for check in checklist.checklist:
    print(check)    

checklist=[EvaluationCheck(check_name='instructions_follow', justification='The assistant provided a thorough answer with definitions and guidelines, and included citations in the required format.', check_pass=True), EvaluationCheck(check_name='instructions_avoid', justification='No disallowed content; no extraneous actions.', check_pass=True), EvaluationCheck(check_name='answer_relevant', justification='Directly answers what a dbt model is and how it works.', check_pass=True), EvaluationCheck(check_name='answer_clear', justification='Clear, structured explanation with bullet points.', check_pass=True), EvaluationCheck(check_name='answer_citations', justification='Citations included with GitHub links to source material.', check_pass=True), EvaluationCheck(check_name='completeness', justification='Covers materialization, SQL files, dependencies, configurations, execution, version control, and documentation; provides example structure.', check_pass=True), EvaluationCheck(check_name='tool

In [52]:
question_generation_prompt = """
You are helping to create test questions for an AI agent that answers questions about DBT.

Based on the provided DBT content, generate realistic questions that students might ask.

The questions should:

- Be natural and varied in style
- Range from simple to complex
- Include both specific technical questions and general course questions

Generate one question for each record.
""".strip()

class QuestionsList(BaseModel):
    questions: list[str]

question_generator = Agent(
    name="question_generator",
    instructions=question_generation_prompt,
    model='gpt-4o-mini',
    output_type=QuestionsList
)


In [57]:
import random
'''
sample = random.sample(dbt_docs, 10)
prompt_docs = [d['content'] for d in sample]
prompt = json.dumps(prompt_docs)

result = await question_generator.run(prompt)
questions = result.output.questions
'''
questions = []

sample = random.sample(dbt_docs, 10)

for doc in sample:
    prompt = doc['content']
    result = await question_generator.run(prompt)
    questions.append(result.output.questions[0])


In [58]:
for a in questions:
    print(a)

What naming convention should be used for primary keys in DBT models?
What is the purpose of the catalog feature in dbt?
What information does dbt track about each model build?
What is Jinja and how does it enhance SQL in dbt projects?
What are the three main materializations in DBT that cover most analytics engineering needs?
What is a semantic model in the context of DBT?
What is the purpose of enabling source data freshness snapshots in dbt?
What are the different types of materializations available in dbt?
What are incremental models in DBT, and how do they differ from traditional models?
What is the purpose of using `dbt clone` in my CI job?


In [59]:
from tqdm.auto import tqdm

for q in tqdm(questions):
    print(q)

    result = await agent.run(user_prompt=q)
    print(result.output)

    log_interaction_to_file(
        agent,
        result.new_messages(),
        source='ai-generated'
    )

    print()


  0%|          | 0/10 [00:00<?, ?it/s]

What naming convention should be used for primary keys in DBT models?
In dbt models, the naming convention for primary keys is to declare them using the `type` parameter within an `entity` block at the column level. Specifically, you need to specify the `type` as `primary` for any column that serves as a primary key. 

Here’s a brief example of how the syntax looks in a dbt model:

```yaml
columns:
  - name: booking_id
    entity:
      type: primary
      name: booking_id
```

This declaration indicates that `booking_id` is the primary key for the table or model. If your model contains a primary entity, you can also set the top-level `primary_entity` to name the model’s primary entity.

It's important to ensure that the primary key is unique for each record in the model, contributing to data integrity and proper relationships.

For more detailed information on this, you can refer to the section on [Semantic models](https://github.com/dbt-labs/docs.getdbt.com/blob/current/website/docs/

In [69]:
eval_set = []

for log_file in LOG_DIR.glob('*.json'):
    if 'dbt_agent_v2' not in log_file.name:
        continue

    log_record = load_log_file(log_file)
    if log_record['source'] != 'ai-generated':
        continue

    eval_set.append(log_record)


In [71]:
eval_results = []

for log_record in tqdm(eval_set):
    eval_result = await evaluate_log_record(eval_agent, log_record)
    eval_results.append((log_record, eval_result))


  0%|          | 0/10 [00:00<?, ?it/s]

In [72]:
rows = []

for log_record, eval_result in eval_results:
    messages = log_record['messages']

    row = {
        'file': log_record['log_file'].name,
        'question': messages[0]['parts'][0]['content'],
        'answer': messages[-1]['parts'][0]['content'],
    }

    checks = {c.check_name: c.check_pass for c in eval_result.checklist}
    row.update(checks)

    rows.append(row)


In [73]:
import pandas as pd

df_evals = pd.DataFrame(rows)



In [74]:
print(len(eval_results))
df_evals.mean(numeric_only=True)


10


Series([], dtype: float64)

In [79]:
print(df_evals['instructions_follow'].unique())

[nan True False]


In [80]:
eval_cols = [
    'instructions_follow',
    'instructions_avoid',
    'answer_relevant',
    'answer_clear',
    'answer_citations',
    'completeness',
    'tool_call_search'
]

df_scores = df_evals.copy()

df_scores[eval_cols] = df_scores[eval_cols].astype(float)

print(df_scores[eval_cols].mean())

instructions_follow    0.888889
instructions_avoid     1.000000
answer_relevant        0.875000
answer_clear           0.875000
answer_citations       0.875000
completeness           0.875000
tool_call_search       1.000000
dtype: float64
